# Solutions · Chapter 04-01 · Framing: turning a request into a task

Worked answers for `notebooks/04_workflow/04-01_framing.ipynb`.

Several of these are questions with no single right answer, marked on whether the reasoning holds. Where
that is the case it is said so.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# SYNTHETIC. The chapter's gym panel, rebuilt.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members()
print("%d rows, %d members" % (len(panel), panel.member_id.nunique()))

## Quick understanding

### E1

1. **Unit of observation** - what is one row?
2. **Target** - what exactly is being predicted, as a column you can construct?
3. **Prediction time** - at what moment is the prediction made?
4. **Horizon** - how far past that moment does the answer look?
5. **Availability** - what is known at the moment of prediction?

Plus the three that make them answerable: **population**, **what good looks like as a number**, and
**the decision this feeds**.

### E2

**Unit of observation** decides what one row is, and therefore every denominator, every average, and
what the model can be asked at prediction time. It is why "the churn rate" here is 33.00%, 2.19% or
1.51% depending on nothing but this choice.

**Horizon** decides how far past the wall the target looks. It sets the base rate, and through the base
rate it sets what any metric means - and it determines which features can possibly be useful, since
next-month behaviour and next-year behaviour are driven by different things.

### E3

Because they are not eligible for the prediction. **A model that runs at the end of month 12 runs on
members who are still members** - you cannot make a retention offer to somebody who left in month 7.

Labelling them 0 would be a lie in two directions at once: it says "did not cancel in months 13-18",
which is true only in the sense that they had already gone, and it inflates the denominator with rows
that will never appear in production. The base rate would fall, the metric would improve, and the model
would spend its capacity learning to identify the already-departed - a group that is trivially
identifiable and completely useless.

**Defining the population is part of framing, and "everybody in the table" is almost never it.**

## Hand calculation

### E4

**(a) Per member.** 250 of 1,000 members cancel in the year: **25.00%**.

**(b) Per member-month.** Count the denominator properly:

- 750 members survive the year and contribute 12 months each: `750 x 12` = 9,000
- 250 members cancel and contribute 6 months on average: `250 x 6` = 1,500
- Total member-months: **10,500**

There are still only 250 cancellations, so the rate is `250 / 10500` = **2.38%**.

**Why they differ.** The numerator is identical - the same 250 events. Only the denominator changed, from
1,000 people to 10,500 opportunities. The per-member rate answers "what fraction of people leave in a
year"; the per-member-month rate answers "in any given month, what fraction of active memberships end".

Note also that the two are not related by a simple factor of 12: `25% / 12` is 2.08%, not 2.38%, because
members who cancel contribute fewer months to the denominator than members who stay. **Whenever the unit
is a period of exposure, the entities that experience the event are under-represented in the
denominator** - which is exactly why survival and hazard analysis exist as separate subjects.

In [ ]:
survivors, leavers = 750, 250
member_months = survivors * 12 + leavers * 6
print("per member      : %d / %d = %.2f%%" % (leavers, survivors + leavers,
                                              100 * leavers / (survivors + leavers)))
print("member-months   : %d x 12 + %d x 6 = %d" % (survivors, leavers, member_months))
print("per member-month: %d / %d = %.2f%%" % (leavers, member_months, 100 * leavers / member_months))
print()
print("the naive 'divide by 12' answer: %.2f%% - wrong, and wrong in a predictable direction"
      % (100 * leavers / (survivors + leavers) / 12))

### E5

From the chapter's horizon table: the base rate is **3.84%** at two months and **18.23%** at nine.
"Nobody leaves" therefore scores:

- **two months: 96.16%**
- **nine months: 81.77%**

Those are the numbers a model must **beat**, and beating them is not the same as being useful. To have
demonstrated anything at all a model needs to exceed them by a margin larger than the uncertainty in the
estimate - and at a two-month horizon there are only 20 cancellers in 521 members, so a handful of
lucky calls moves the accuracy by a percentage point.

The honest statement of the requirement is therefore not "beat 96.16%" but **"beat 96.16% by more than
the noise, on data not used to fit"** - which is why 04-02 and 04-03 come next in that order.

### E6

- **Offers wasted:** 60 offers, 22 of them to real cancellers, so `38 / 60` = **63.33%** are wasted.
- **Cancellers reached:** `22 / 70` = **31.43%**.

**Finance will ask about the first.** The 63.33% is the line in their budget - it is money spent on
people who were going to stay anyway. Retention will ask about the second, because two-thirds of the
people they were trying to save were never contacted.

Both numbers are correct, they measure different failures, and **they trade against each other**: fund
more offers and you reach more cancellers while wasting a higher share. This is precision against recall
- 36.67% and 31.43% respectively - meeting you before the chapter that names them (06-05). It arrives
here because it is a framing question: **which of the two errors costs more is decided by the business,
before any model exists.**

### E7

A member who joins in month 3 and cancels in month 15, evaluated at a wall of month 12:

- **`months_on_file` = 13** (months 3 to 15 inclusive)
- **`tenure_months` = 10** (months 3 to 12 inclusive)

Only `tenure_months` is usable, because **at the end of month 12 the member has not cancelled yet**, so
13 is not a number anybody could have written down at that moment. It exists only because we are looking
at the table after the fact.

The giveaway is that the two differ *at all*. If a "tenure" column computed at the wall disagrees with
one computed over the whole table, the second one has seen the future.

In [ ]:
print("E6")
offers, hits, cancellers = 60, 22, 70
print("  wasted offers   : %d / %d = %.2f%%" % (offers - hits, offers, 100 * (offers - hits) / offers))
print("  cancellers found: %d / %d = %.2f%%" % (hits, cancellers, 100 * hits / cancellers))
print("  (precision %.2f%%, recall %.2f%%)" % (100 * hits / offers, 100 * hits / cancellers))
print()
print("E7  joined month 3, cancelled month 15, wall at month 12")
print("  months_on_file = 15 - 3 + 1 = %d   <- not knowable at the wall" % (15 - 3 + 1))
print("  tenure_months  = 12 - 3 + 1 = %d   <- knowable" % (12 - 3 + 1))

## Coding

### E8 - the framing function

The whole point is that the wall exists **once**, at the top, and every feature is built from the
filtered table. Passing `history` around rather than `panel` makes the mistake in the failure lab
structurally impossible rather than merely discouraged.

In [ ]:
def frame(panel, cut, horizon):
    active = panel[(panel.month == cut) & (panel.cancelled == 0)].member_id.unique()
    history = panel[panel.member_id.isin(active) & (panel.month <= cut)]     # the wall, once
    future = panel[panel.member_id.isin(active) & (panel.month > cut)]

    left = future[(future.month <= cut + horizon) & (future.cancelled == 1)].member_id.unique()
    target = pd.Series(np.isin(active, left).astype(int), index=active, name="cancels_in_window")

    features = pd.DataFrame({
        "visits_this_month": history[history.month == cut].set_index("member_id").visits.reindex(active),
        "mean_visits_last_3": history[history.month > cut - 3].groupby("member_id").visits.mean().reindex(active),
        "tenure_months": history.groupby("member_id").size().reindex(active),
        "tickets_so_far": history.groupby("member_id").tickets.sum().reindex(active),
    })
    return features, target


features, target = frame(panel, cut=12, horizon=6)
print("members %d, base rate %.2f%%  (chapter: 521 and 13.44%%)" % (len(target), 100 * target.mean()))

### E9 - three walls

In [ ]:
rows = []
for cut in [6, 12, 18]:
    _, t = frame(panel, cut, horizon=6)
    rows.append({"wall (month)": cut, "members eligible": len(t),
                 "cancel within 6 months": int(t.sum()),
                 "base rate": "%.2f%%" % (100 * t.mean())})
print(pd.DataFrame(rows).to_string(index=False))

**263 members at month 6, 521 at month 12, 451 at month 18**, with base rates of 10.65%, 13.44% and
10.86%.

The population is smallest at month 6 because members are still joining - everyone joins in months 1 to
12, so at the month-6 wall only about half of them exist yet. It falls again by month 18 because
cancellations have accumulated. **Neither end is a defect; both are the correct population for their own
wall.**

**Which wall for a model retrained monthly? None of them - all of them.** The question contains its own
answer: if the model runs every month, then every month is a prediction point, and picking one is
choosing to throw away eleven-twelfths of the available training data.

The right construction is E18's: **stack the walls.** Build the problem at every month, concatenate,
and train on all of it. That is what the second half of this module is about, and it brings a new
problem with it - the same member now appears in many rows, so a random train/test split will put the
same person on both sides. Chapter 04-04 is about exactly that.

**The one thing you must not do is choose the wall after seeing which one gives the best score.** The
base rates differ by three percentage points, so there is real room to flatter a model by picking a
month, and doing so silently is indistinguishable from cheating.

### E10 - does the model get better as the horizon grows?

Worth doing carefully, because the obvious way to do it gives a confident wrong answer.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))


single, cross_validated, spread, positives, horizons = [], [], [], [], list(range(2, 13))
for horizon in horizons:
    X, y = frame(panel, 12, horizon)
    train_X, test_X, train_y, test_y = train_test_split(
        X, y, test_size=0.3, random_state=0, stratify=y)
    fitted = model().fit(train_X, train_y)
    single.append(roc_auc_score(test_y, fitted.predict_proba(test_X)[:, 1]))

    folds = cross_val_score(model(), X, y, scoring="roc_auc",
                            cv=StratifiedKFold(5, shuffle=True, random_state=0))
    cross_validated.append(folds.mean())
    spread.append(folds.std())
    positives.append(int(y.sum()))

print(pd.DataFrame({"horizon": horizons, "cancellers": positives,
                    "AUC, one split": np.round(single, 3),
                    "AUC, 5-fold mean": np.round(cross_validated, 3),
                    "fold-to-fold sd": np.round(spread, 3)}).to_string(index=False))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))

left.plot(horizons, single, "o-", color="#D55E00", label="one 70/30 split")
left.plot(horizons, cross_validated, "s-", color="#0072B2", label="5-fold mean")
left.fill_between(horizons, np.array(cross_validated) - np.array(spread),
                  np.array(cross_validated) + np.array(spread), color="#0072B2", alpha=0.15)
left.axhline(0.5, color="#999999", linestyle=":")
left.set_xlabel("horizon (months)")
left.set_ylabel("AUC")
left.set_title("The trend you think you see is the noise")
left.legend(fontsize=8)

right.plot(positives, spread, "o", color="#0072B2")
for horizon, count, sd in zip(horizons, positives, spread):
    if horizon in (2, 4, 6, 12):
        right.annotate("h=%d" % horizon, (count, sd), textcoords="offset points", xytext=(6, 4),
                       fontsize=8)
right.set_xlabel("number of cancellers in the data")
right.set_ylabel("fold-to-fold sd of AUC")
right.set_title("What actually changes: how precisely you can measure")

plt.tight_layout()
plt.show()

**The answer is that it does not systematically change, and the single-split run will tell you
otherwise very convincingly.**

On one 70/30 split the AUC reads 0.845 at a two-month horizon and 0.636 at six - which looks like a
clear, mechanistically plausible story: near-term churn is driven by recent behaviour, which these
features capture, and distant churn is not. It would survive a meeting.

Cross-validated, every horizon sits between **0.698 and 0.733**, and the fold-to-fold standard deviation
at a two-month horizon is **0.149**. The apparent 0.21 difference is smaller than the measurement error
of a single one of the numbers being compared.

**What does change is the precision, and only at the short end.** At a two-month horizon there are 20
cancellers, of whom about 6 land in a test split, and the fold-to-fold spread is 0.149 - five times any
other horizon's. Past about 35 cancellers the spread settles into a band between 0.03 and 0.07 with no
further trend; more positives stop helping once there are enough for each fold to contain a reasonable
number. The right-hand panel is one dramatic point and then a flat scatter, which is the honest shape:
**the uncertainty is governed by the count of the rare class, and it hurts sharply when that count is
small and stops mattering once it is not.** The number of rows - 521 throughout - never enters.

Two things to take from it:

1. **A single split cannot support a claim about a trend.** If you are going to compare eleven models,
   you need eleven estimates with error bars, and the cheapest way to get them is cross-validation
   (04-07).
2. **Count your positives before you interpret anything.** 20 events is not a dataset, whatever the row
   count says. The 521 in the denominator is reassuring and irrelevant.

### E11 - a leakage detector

In [ ]:
def crosses_the_wall(panel, cut, builder):
    # builder(table) -> DataFrame of features. Called twice: once on the whole panel, once
    # on rows at or before the wall. Any column whose values move has seen the future.
    everything = builder(panel)
    only_past = builder(panel[panel.month <= cut])
    guilty = []
    for column in everything.columns:
        left, right = everything[column].align(only_past[column], join="inner")
        if not np.allclose(left.astype(float), right.astype(float), equal_nan=True):
            guilty.append(column)
    return guilty


def build_features(table):
    members = table.member_id.unique()
    return pd.DataFrame({
        "visits_this_month": table[table.month == table.month.max()].set_index("member_id").visits.reindex(members),
        "tenure_months": table[table.month <= 12].groupby("member_id").size().reindex(members),
        "tickets_so_far": table[table.month <= 12].groupby("member_id").tickets.sum().reindex(members),
        "visits_lifetime": table.groupby("member_id").visits.sum().reindex(members),
        "months_on_file": table.groupby("member_id").size().reindex(members),
    }).fillna(0)


print("columns that change when the future is removed:")
for column in crosses_the_wall(panel, 12, build_features):
    print("  -", column)

It flags **`visits_this_month`, `visits_lifetime` and `months_on_file`**, and clears `tenure_months` and
`tickets_so_far`.

The two obvious culprits are caught. `visits_this_month` is flagged because of how it was written -
`table.month == table.month.max()` means "the latest month in whatever table you hand me", which is
month 24 on the full panel and month 12 on the filtered one. **That is a true positive**, and a good
illustration of a real class of bug: a feature defined relative to "the end of the data" rather than to
the wall silently changes meaning the moment the data is extended.

What this detector cannot catch is the combination from the chapter. `tenure_months` is genuinely clean
by this test, and it was still the column that took the leak from 0.954 to 1.000. **A per-column test
catches per-column leaks.** Use it as a cheap filter, not as a guarantee - the guarantee comes from
building features from a filtered table in the first place.

## Interpretation

### E12

In this order, because each one can end the conversation before the next matters:

1. **"Accurate at predicting what, exactly - and over what horizon?"** If the target turns out to be
   "ever cancels", 94% is a different claim entirely.
2. **"What does predicting the majority class score?"** If the base rate is 13%, the answer is 87% and
   the model has bought 7 points. If the base rate is 6%, the constant scores 94% and the model has
   bought nothing at all.
3. **"Measured on which rows?"** If it is the rows it was fitted on, the number is not evidence
   (module 03's assessment demonstrated a model beating the noise it was built from).
4. **"Which feature is doing the work, and when is its value known?"** Ask for the strongest coefficient
   or importance and apply the wall test to it.

Only then is it worth reading code. Three of the four questions can be answered by the person who built
it, in a corridor, in under a minute - and in my experience one of the first two ends it more often than
not.

### E13

You would need: **the base rate**, **the horizon**, **the population**, and **whether the number is from
held-out data**. Without all four, 91% is not a claim about anything.

A plausible way for it to be true and worthless at the same time: the vendor defines churn as
cancellation **within twelve months**, evaluates on **all customers including those who have already
left**, and reports **accuracy**. If 91% of the population does not cancel in that window, then a model
that always says "will not cancel" scores exactly 91% - and it is a genuinely accurate model that
identifies no churner whatsoever. Nothing in the sentence is a lie.

This is not a hypothetical about dishonest vendors. It is the default outcome of using accuracy on an
imbalanced problem, and it is why the contract in this chapter has a row for what good looks like.

## Debugging

### E14

**Cause 1: leakage.** A feature exists in the development table that cannot be computed at prediction
time - the chapter's `months_on_file`. Development sees the future; production does not.

**Cause 2: a train/test split that shares information across the boundary.** The same entity, or the
same time period, appears on both sides, so the "held-out" score was never held out. Development
measured memorisation; production cannot memorise unseen customers.

**How to tell them apart, cheaply and in order:**

- **Refit with each feature dropped in turn.** If one column's removal collapses development performance
  to something near the production number, that column is the leak. This is one loop and it usually ends
  the investigation.
- **Re-split by entity or by time and re-score, changing nothing else.** If development performance
  collapses to the production number when the split respects groups or chronology, the split was the
  problem. If it stays at 0.998, it is not.

The two causes are distinguished by which intervention moves the number, and both interventions are a
few lines. Note that both diagnoses have the same shape: **make development resemble production, and see
whether the score survives.** That is the entire method, and every technique in the rest of this module
is a way of doing it by default rather than by remembering.

## Exam and interview reasoning

### E15

> "First: what is one row? For churn, one row per active member per month, not one per member - a member
> who has been with us three years should be three years' worth of prediction opportunities, not one.
> Second: the target has to be a column I can build, so 'cancels within the next N months of this
> month', not 'is likely to leave'. Third: the prediction point - a wall. Everything before it is
> features, everything after is the answer. Fourth: the horizon, N, which I take from the retention
> team's campaign schedule, because a prediction they cannot act on within the window is not worth
> making. Fifth: every feature is computed from rows before the wall, which in practice means filtering
> the table once, up front, and never touching the raw table again. Then: the population is members
> still active at the wall; the metric is ranking quality rather than accuracy, because at a 13% base
> rate a constant scores 87%; and the decision it feeds is which N members get an offer, where N is the
> budget."

**The follow-up: why not predict whether they will ever leave?** Because everyone eventually leaves, so
the target is either trivially 1 or defined by how long you happened to observe them - it is a question
about the length of your data, not about the member. It also has no prediction time and no horizon, so
there is no moment at which to act and no way to be wrong: a member who has not left yet is not a
counter-example, they just have not left *yet*. **A target that cannot be falsified within a stated
window is not a prediction task.**

## Transfer to a different situation

### E16

| # | Question | Answer for hospital readmission |
|---|---|---|
| 1 | **Unit** | One discharge. Not one patient - a patient can be discharged several times, and each is a separate opportunity |
| 2 | **Target** | 1 if the patient is admitted again within the horizon, else 0. Requires deciding whether planned readmissions, transfers and admissions to other hospitals count |
| 3 | **Prediction time** | The moment of discharge. Possibly also 24 hours before, if the decision it feeds is discharge planning rather than follow-up |
| 4 | **Horizon** | 30 days is the usual reporting standard, but the right answer comes from the intervention - a follow-up call programme that runs weekly implies something different from a home-visit service |
| 5 | **Availability** | Everything recorded up to discharge. Nothing from the readmission itself, and nothing back-filled later |
| | **Population** | Discharges alive, excluding those to hospice or another facility, depending on the target |
| | **What good looks like** | Better ranking than the existing clinical judgement, by enough to justify the follow-up capacity |
| | **The decision** | Which discharges get enhanced follow-up, limited by nursing hours |

**A column that would certainly cross the wall:** a **discharge summary or coded diagnosis finalised
after the fact**. Clinical coding is often completed days or weeks after discharge and may be revised in
light of what happened next, so a diagnosis code in the historical table can encode the readmission it is
supposed to predict. Others in the same family: the discharge destination when it is updated later, any
"total length of stay" that spans the readmission, and mortality flags.

Credit for anything with that structure: **a field whose recorded value was written, or revised, after
the moment of prediction.**

## Explain it to someone non-technical

### E17

> "The model has to answer a question with a deadline. Not 'is this member unhappy' but 'will this
> member leave before your next campaign goes out'. Those are different questions and they need
> different models, so I need to know when your campaigns run before I can build one. If you run them
> twice a year, I build something that looks six months ahead - and it will flag more people than a
> next-month version would, because more can happen in six months. If you tell me the schedule after
> the model is built, I have to build it again."

(97 words. It contains the horizon, why it is a business input rather than a technical one, its effect on
the size of the flagged list, and the cost of deciding late.)

## Optional challenge

### E18 - the member-month framing

In [ ]:
def frame_all_months(panel, horizon, first=1, last=18):
    pieces = []
    for cut in range(first, last + 1):
        features, target = frame(panel, cut, horizon)
        if len(target) == 0:
            continue
        block = features.copy()
        block["cut_month"] = cut
        block["target"] = target
        pieces.append(block.reset_index(names="member_id"))
    return pd.concat(pieces, ignore_index=True)


stacked = frame_all_months(panel, horizon=6)
print("rows            : %d   (against 521 for the single wall)" % len(stacked))
print("distinct members: %d" % stacked.member_id.nunique())
print("rows per member : %.1f on average, up to %d"
      % (len(stacked) / stacked.member_id.nunique(), stacked.member_id.value_counts().max()))
print("base rate       : %.2f%%" % (100 * stacked.target.mean()))
print()
print("how often the same member appears:")
print(stacked.member_id.value_counts().describe()[["min", "50%", "max"]].round(1).to_string())

**6,318 rows from 580 members - about 11 rows each - at a base rate of 11.98%.**

Twelve times the training data, and it is the right construction: in production this model runs every
month, so every month is a genuine prediction opportunity and training on one arbitrary month throws
most of the evidence away.

**The new problem is that the same member now appears about eleven times.** Those rows are not
independent. Member 314 in month 7 and member 314 in month 8 share almost all of their history, most of
their features, and often their label.

That breaks a random train/test split completely. Put month 7 in training and month 8 in test, and the
"held-out" score is measuring whether the model can recognise a member it has already seen - which it
can, easily, and which tells you nothing about a member it has not. The result is an optimistic score
that will not survive deployment, produced by code containing no bug at all.

**The fix is to split by member, not by row**, so that every row belonging to one person lands on the
same side. That is **grouped splitting**, and it is chapter 04-04. It is worth noticing that the fix is
forced by a decision made here, in framing: the moment you chose one row per member-month, you chose a
splitting strategy too.

There is a second, subtler issue for later: the rows also overlap in **time**, since a target measured
over months 8-13 and one measured over months 9-14 share five months of future. Even a member-grouped
split leaves that in place, and it is the reason time-based problems get their own splitting rules in the
same chapter.